In [ ]:
-- Распределение номиналов по месяцам

with prepared as (

    select
        client_id,

        date_trunc('month', mrc_start_date)::date as month_dt,

        case
            when lower(com_cus_sgr_desc) like '%200%' then 200
            when lower(com_cus_sgr_desc) like '%300%' then 300
            when lower(com_cus_sgr_desc) like '%400%' then 400
            when lower(com_cus_sgr_desc) like '%500%' then 500
            else null
        end as nominal_amount

    from cvm_sbx.YOUR_TABLE

    where client_id is not null
),

month_nominal_distribution as (

    select
        month_dt,
        nominal_amount,

        count(distinct client_id) as clients_cnt

    from prepared

    where nominal_amount is not null

    group by
        month_dt,
        nominal_amount
)

select
    month_dt,
    nominal_amount,
    clients_cnt,

    round(
        clients_cnt * 100.0 /
        sum(clients_cnt) over (partition by month_dt),
        2
    ) as nominal_share_pct

from month_nominal_distribution

order by
    month_dt,
    nominal_amount;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_nominal_dist['month_dt'] = pd.to_datetime(
    df_nominal_dist['month_dt']
)

pivot_df = (
    df_nominal_dist
    .pivot(
        index='month_dt',
        columns='nominal_amount',
        values='nominal_share_pct'
    )
    .fillna(0)
)

pivot_df = pivot_df.sort_index()

pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6)
)

plt.title('Распределение номиналов по месяцам')
plt.xlabel('Месяц')
plt.ylabel('Доля клиентов, %')

plt.legend(
    title='Номинал',
    bbox_to_anchor=(1.02, 1)
)

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()